# 05 · Co psuje porównanie?
**Cel:** zobaczyć wpływ głośności i tła, zanim uznamy różnicę za defekt. Zmodyfikowane wersje jednej płytki to nadal jedna płytka.

In [1]:
from pathlib import Path
import os, json, sys
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Otwórz cały folder projektu w VS Code."
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
from pinglab.audio import Config, load_audio, detect_hits, cut_hits, listen
from pinglab.session import load_session
SESSION = load_session(ROOT)
CONFIG, FMAX = SESSION.config, SESSION.fmax
MANIFEST, ANNOTATIONS = SESSION.manifest, SESSION.annotations
print("Dane:", SESSION.source_kind, "· manifest:", MANIFEST.relative_to(ROOT) if MANIFEST.is_relative_to(ROOT) else MANIFEST.name)
print("Wspólne pasmo do", FMAX, "Hz · konfiguracja: project.json")
if SESSION.source_kind == "synthetic":
    display(Markdown("**SYMULACJA — wynik sprawdza narzędzia, nie wykrywanie pęknięć u klienta.**"))


Dane: synthetic · manifest: data/demo/manifest.csv
Wspólne pasmo do 20000.0 Hz · konfiguracja: project.json


**SYMULACJA — wynik sprawdza narzędzia, nie wykrywanie pęknięć u klienta.**

In [2]:
# KONFIGURACJA
FILE = Path(SESSION.pair()[0].resolved_path)
BACKGROUND = SESSION.background  # np. ROOT / "data/raw/factory_background.wav"
audio=load_audio(FILE,CONFIG)
clips=cut_hits(audio.y,audio.sr,detect_hits(audio.y,audio.sr,CONFIG),CONFIG)
assert clips,"Popraw segmentację w notebooku 01/02."
clip=clips[0]
noise=load_audio(BACKGROUND,CONFIG).y if BACKGROUND else None

## Kontrolowane zmiany
Bez pliku tła dodajemy syntetyczny biały szum. To test techniczny; prawdziwa fabryka ma inne widmo i zmienność. SNR = stosunek energii sygnału do tła w tym oknie, nie pomiar poziomu hałasu hali.

In [3]:
from pinglab.robustness import sensitivity_table
from pinglab.plots import compare_hits
stats,variants=sensitivity_table(clip['y'],audio.sr,noise,clip['onset_offset_s'],FMAX)
display(stats[['variant','peak_hz','centroid_hz','decay10_s','flatness','clipping_fraction']].round(3))
display(compare_hits([variants['original'],variants['noise 10 dB']],audio.sr,
                     ['Oryginał','Szum 10 dB'],[clip['onset_offset_s']]*2,FMAX))
for name in ['original','gain -12 dB','noise 10 dB']:
    print(name)
    # Normalize only for convenient playback; metrics above use raw perturbations.
    display(listen(variants[name],audio.sr,normalize=True))

,variant,peak_hz,centroid_hz,decay10_s,flatness,clipping_fraction
0,original,1804.688,2219.773,0.24,0.000,0.0
1,gain -12 dB,1804.688,2219.773,0.24,0.000,0.0
2,noise 20 dB,1804.688,2297.123,0.24,0.010,0.0
3,noise 10 dB,1804.688,2932.735,0.24,0.093,0.0
4,noise 0 dB,1804.688,6184.278,0.44,0.505,0.0


original


gain -12 dB


noise 10 dB


**Oczekiwane:** zmiana gainu prawie nie zmienia cech kształtu. Szum skraca wiarygodny fragment zaniku i zmienia barwę. Nie odszumiaj agresywnie „na wszelki wypadek”: możesz usunąć sygnał wady. Lepiej poprawić nagranie i porównać wyniki przed / po świadomie dobranym filtrowaniu.

## Czy model przeżywa zmianę tła?
Ten test zostawia całe płytki poza treningiem. Porównuje trening na czystym audio i z dodanym tłem; powielone warianty nie zwiększają liczby płytek. Dane i tło pochodzą ze wspólnego `project.json`. Zatwierdź wcześniej czasy w 02. Biały szum nie odtwarza całej hali.

In [4]:
from pinglab.dataset import analyze_manifest
from pinglab.robustness import noise_robustness
STRESS_MANIFEST = SESSION.manifest
STRESS_ANNOTATIONS = SESSION.annotations
stress_events, _ = analyze_manifest(STRESS_MANIFEST,SESSION.output/'stress',CONFIG,FMAX,
    STRESS_ANNOTATIONS if STRESS_ANNOTATIONS.exists() else None)
try:
    stress = noise_robustness(stress_events, family='resonance', noise=noise, config=CONFIG, fmax=FMAX, snrs=(20,10,5,0))
    display(stress.round(3))
    stress.to_csv(SESSION.output/'stress/robustness.csv',index=False)
except ValueError as error:
    print(error)

,training,test_variant,balanced_accuracy,recall_nok,false_alarm_rate,n_plates,family
0,clean only,original,0.975,1.0,0.05,30,resonance
1,clean only,noise_20dB,1.000,1.0,0.00,30,resonance
2,clean only,noise_10dB,0.500,0.0,0.00,30,resonance
3,clean only,noise_5dB,0.500,0.0,0.00,30,resonance
4,clean only,noise_0dB,0.500,0.0,0.00,30,resonance
5,clean + noise,original,0.950,1.0,0.10,30,resonance
6,clean + noise,noise_20dB,0.950,1.0,0.10,30,resonance
7,clean + noise,noise_10dB,0.950,1.0,0.10,30,resonance
8,clean + noise,noise_5dB,0.950,1.0,0.10,30,resonance
9,clean + noise,noise_0dB,1.000,1.0,0.00,30,resonance


## Protokół na pierwszą serię
1. Zapisz ID płytki, ocenę operatora, typ i miejsce wady; niepewne etykiety oznacz `unknown`.
2. Ustal podparcie, mikrofon, gain i kolejność miejsc uderzeń. Nie zmieniaj ich między OK i NOK.
3. Preferuj WAV PCM 48 kHz / 24 bit. Zostaw zapas amplitudy, wyłącz AGC i odszumianie komunikatora.
4. Nagraj chwilę ciszy przed uderzeniem i pełne wybrzmiewanie po nim; odstęp ustal na pierwszym odsłuchu.
5. Zbierz powtórzenia na różnych płytkach obu klas i osobne tło hali. Przeplataj OK i NOK, by czas/sesja nie zdradzały etykiety.
6. Jeśli operator wskazuje jedyne słyszalne miejsce wady, zaznacz ten konkretny hit. Mediana z dziesięciu miejsc może go ukryć.
7. Osobno sprawdź zmianę miejsca mikrofonu, siły uderzenia i hałasu. Nie mieszaj od razu wszystkich zmian.

## Decyzja po pierwszych 30 płytkach
**Kontynuuj**, jeżeli różnica powtarza się na niewidzianych detalach i ma sens akustyczny. **Popraw protokół**, jeśli separacja pokrywa się z sesją, głośnością lub podparciem. **Zbierz więcej danych**, jeśli błędy zależą od konkretnego typu wady. Uzgodnij z klientem osobno dopuszczalne przeoczenia NOK i odrzucenia OK.

Nie ma metody niezależnej od dowolnego nagrania. Ten zestaw pomaga szybko ustalić, co jest stabilne i czego jeszcze brakuje.